# 高级分类模型

本示例展示了如何使用更高级的分类器来替代默认使用的线性分类器。

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

from reservoir_computing.modules import RC_model
from reservoir_computing.utils import compute_test_scores
from reservoir_computing.datasets import ClfLoader

np.random.seed(0) # 固定随机种子以确保可重复性

## 准备数据

我们将使用数据加载器 `ClfLoader` 来获取分类数据集。
要查看有哪些可用的数据集，我们可以调用函数 `available_datasets`。通过设置 `details=True` 可以获取额外信息。

In [ ]:
downloader = ClfLoader()
downloader.available_datasets(details=False)  # 描述可用的数据集

Available datasets:

AtrialFibrillation
ArabicDigits
Auslan
CharacterTrajectories
CMUsubject16
ECG2D
Japanese_Vowels
KickvsPunch
Libras
NetFlow
RobotArm
UWAVE
Wafer
Chlorine
Phalanx
SwedishLeaf


接下来，我们加载表示九位不同说话者发音的不同日语音的 MTS 数据集。目标是正确分类说话者。注意我们需要将标签转换为独热编码向量。

In [4]:
Xtr, Ytr, Xte, Yte = downloader.get_data('Japanese_Vowels')

Loaded Japanese_Vowels dataset.
Number of classes: 9
Data shapes:
  Xtr: (270, 29, 12)
  Ytr: (270, 1)
  Xte: (370, 29, 12)
  Yte: (370, 1)


In [ ]:
# 标签的独热编码
onehot_encoder = OneHotEncoder(sparse_output=False)
Ytr = onehot_encoder.fit_transform(Ytr)
Yte = onehot_encoder.transform(Yte)

然后，我们定义储层、降维模块和多元时间序列（MTS）表示类型的配置。

In [ ]:
config = {}

# 储层的超参数
config['n_internal_units'] = 450        # 储层的大小
config['spectral_radius'] = 0.59        # 储层的最大特征值
config['leak'] = 0.6                    # 储层状态更新中的泄漏量（None 或 1.0 表示无泄漏）
config['connectivity'] = 0.25           # 储层中非零连接的百分比
config['input_scaling'] = 0.1           # 输入权重的缩放
config['noise_level'] = 0.01            # 储层状态更新中的噪声
config['n_drop'] = 5                    # 要丢弃的瞬态状态数
config['bidir'] = True                  # 如果为 True，使用双向储层
config['circle'] = False                # 使用圆形拓扑的储层

# 降维超参数
config['dimred_method'] = 'tenpca'      # 选项：{None（无降维）, 'pca', 'tenpca'}
config['n_dim'] = 75                    # 降维过程后的结果维度数

# MTS 表示类型
config['mts_rep'] = 'reservoir'         # MTS 表示：{'last', 'mean', 'output', 'reservoir'}
config['w_ridge_embedding'] = 10.0      # 岭回归的正则化参数

## 线性读出层

我们将开始使用简单的线性分类器作为读出层。具体来说，我们将使用 sklearn 的 [RidgeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeClassifier.html)。该分类器需要定义一个正则化参数，我们称之为 `w_ridge`（但在 sklearn 中称为 `alpha`）。

In [ ]:
# 读出层类型
config['readout_type'] = 'lin'          # 用于分类的读出层
config['w_ridge'] = 1.0                 # 岭回归读出层的正则化

此时，我们通过传递之前指定的配置来初始化 RC 分类器，然后在训练数据上拟合它。

In [ ]:
classifier =  RC_model(**config)

# 训练模型
tr_time = classifier.fit(Xtr, Ytr) 

Training completed in 0.01 min


此时，我们可以预测测试集的标签，并通过计算分类准确率和 F1 分数来查看它们与真实标签的相似程度。

In [ ]:
# 计算测试数据的预测
pred_class = classifier.predict(Xte) 
accuracy, f1 = compute_test_scores(pred_class, Yte)
print(f"Accuracy = {accuracy:.3f}, F1 = {f1:.3f}")

Accuracy = 0.973, F1 = 0.973


这是一个相当高的准确率。即使是像 RidgeClassifier 这样的简单模型，由于 RC 模型提供的强大表示能力，也能几乎完美地分类测试数据。

接下来，我们将尝试比 RidgeClassifier 更强大的分类器。在这个例子中，由于分类性能已经很高，我们不期望看到性能的极端变化。然而，在更复杂的任务中，使用更强大的分类器可以带来显著的好处。

## 支持向量分类器读出层

我们将从 sklearn 的支持向量机分类器 [SVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) 开始。

首先需要定义新分类器的超参数，并将它们传递给 RC 模型。

In [ ]:
# 读出层类型
config['readout_type'] = 'svm'          # 用于分类的读出层
config['svm_gamma'] = 5e-3              # RBF 核的带宽
config['svm_C'] = 10.0                  # SVM 超平面的正则化

接下来，我们重新创建 RC 模型，训练它，然后测试它。

In [ ]:
classifier =  RC_model(**config)

# 训练模型
tr_time = classifier.fit(Xtr, Ytr) 

# 计算测试数据的预测
pred_class = classifier.predict(Xte) 
accuracy, f1 = compute_test_scores(pred_class, Yte)
print(f"Accuracy = {accuracy:.3f}, F1 = {f1:.3f}")

Training completed in 0.01 min
Accuracy = 0.954, F1 = 0.955


正如预期的那样，性能仍然很好，但与之前得到的结果没有太大差异。

## 多层感知器读出层

接下来，我们可以使用简单的神经网络作为分类器。我们将使用 sklearn 的多层感知器（[MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)）。

在这种情况下，我们有更多的超参数需要调整。在处理实际应用时，为了找到最优的超参数，您应该使用验证集进行适当的超参数搜索。

In [ ]:
# 读出层类型
config['readout_type'] = 'mlp'          # 用于分类的读出层
config['mlp_layout'] = (64,32)          # 每个 MLP 层中的神经元数
config['num_epochs'] = 2000             # 训练轮数
config['w_l2'] = 1e-4                   # L2 正则化的权重
config['nonlinearity'] = 'tanh'         # 激活函数类型：{'relu', 'tanh', 'logistic', 'identity'}

与之前一样，我们创建 RC 分类器，训练它并在未见过的数据上测试。

In [ ]:
classifier =  RC_model(**config)

# 训练模型
tr_time = classifier.fit(Xtr, Ytr) 

# 计算测试数据的预测
pred_class = classifier.predict(Xte) 
accuracy, f1 = compute_test_scores(pred_class, Yte)
print(f"Accuracy = {accuracy:.3f}, F1 = {f1:.3f}")

Training completed in 0.11 min
Accuracy = 0.959, F1 = 0.961


在这种情况下，分类器也获得了良好的性能，但与之前的情况没有太大差异。

更复杂的模型（如 SVC 和 MLP）需要适当的调优，但在困难的任务中，与简单的线性分类器相比，可以获得更好的性能。